# Direct base-model tools — no adapter
Choose **GPU T4 x2**, enable **Internet**, and attach the private dataset made from **tool_use_source.zip**. Set MODEL_SIZE below. No old contract adapter, selective-facts run, GitHub push or earlier result bundle is required.

Both arms use NF4, the same prompt, JSON tool protocol, bounded tool loop and open-development tasks. Calls are chosen by the model, not automatically inserted. This is custom JSON tool use, not native function calling.

This notebook evaluates the **intent parser**, not the six end-to-end Stage 2 systems. It scores 80 synthetic and 20 existing development tasks and records actual tool calls, correct-reference acceptance and wrong-output acceptance. These are development diagnostics, not an independent test benchmark.

The training notebook first runs a two-step smoke job, then fresh two-epoch QLoRA training. It evaluates the unadapted base and both epoch checkpoints and selects the best candidate under the documented semantic screening rule. A completed training job does not guarantee an eligible adapter. Only the adapter evaluation notebook requires the NEW tool adapter dataset.

Download the output ZIP and executed notebook. Do not run both sizes in the same output directory or attach an old contract/selective-facts adapter here.


In [ ]:
MODEL_SIZE = "1_5b"  # choose "1_5b" or "3b"; use the same size for a fair comparison
assert MODEL_SIZE in ("1_5b", "3b")


In [ ]:
import hashlib,json,os,subprocess,sys,zipfile
from pathlib import Path
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
MANIFEST_SHA='4d0f5ffa0f7464c17451f3d5e5112fd0d6433a5b3e22070e543d9aace4ce8d03'
INPUT=Path('/kaggle/input')
MANIFEST_NAME='tool_use_manifest.json'
ROOT=WORK/('tool_use_source_'+MANIFEST_SHA[:12])

def write_source(manifest_bytes,read_source):
    assert hashlib.sha256(manifest_bytes).hexdigest()==MANIFEST_SHA, 'Wrong source/data bundle version'
    manifest=json.loads(manifest_bytes)
    assert manifest['experiment']=='tool-use-v2'
    for name,expected in manifest['sha256'].items():
        relative=Path(name)
        assert not relative.is_absolute() and '..' not in relative.parts, 'Invalid source path'
        content=read_source(name)
        assert hashlib.sha256(content).hexdigest()==expected, f'Changed source/data: {name}'
        path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert path.read_bytes()==content, f'Changed working copy: {name}'
        else:path.write_bytes(content)
    print('Verified source and data:',len(manifest['sha256']),'files',MANIFEST_SHA)

# Kaggle can expose uploaded ZIPs as files or unpacked dataset directories. Support both.
manifest_paths=[p for p in INPUT.rglob(MANIFEST_NAME) if hashlib.sha256(p.read_bytes()).hexdigest()==MANIFEST_SHA]
if manifest_paths:
    manifest_path=manifest_paths[0]
    write_source(manifest_path.read_bytes(),lambda name:(manifest_path.parent/name).read_bytes())
else:
    archives=list(INPUT.rglob('tool_use_source.zip'))
    found=False
    for archive in archives:
        with zipfile.ZipFile(archive) as z:
            assert len(z.namelist())==len(set(z.namelist())), 'Duplicate source ZIP members'
            name='tool_use_source/'+MANIFEST_NAME
            if name not in z.namelist():continue
            manifest_bytes=z.read(name)
            if hashlib.sha256(manifest_bytes).hexdigest()!=MANIFEST_SHA:continue
            write_source(manifest_bytes,lambda relative:z.read('tool_use_source/'+relative))
            found=True;break
    assert found, 'Attach the private dataset made from tool_use_source.zip. No original adapter is required.'
PACKAGES=WORK/'tool_use_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','finetune.tool_use.preflight','--size',MODEL_SIZE],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.__version__=='2.11.0+cu130'; assert torch.cuda.device_count()==2; assert all('T4' in torch.cuda.get_device_name(i) for i in (0,1)); print('T4 x2 preflight passed')"],cwd=ROOT,env=ENV,check=True)
for physical_gpu in ['0','1']:
    subprocess.run([sys.executable,'-m','finetune.selective_facts.attention'],cwd=ROOT,env=dict(ENV,CUDA_VISIBLE_DEVICES=physical_gpu),check=True)


In [ ]:
def execute(smoke):
    out=WORK/('tool_use_base_'+MODEL_SIZE+('_smoke' if smoke else ''))
    assert not out.exists(), 'Use a fresh session: existing results will not be overwritten.'
    command=[sys.executable,'-m','finetune.tool_use.evaluate','--size',MODEL_SIZE,'--out',str(out),'--packages',str(PACKAGES)]
    if smoke:command+=['--smoke']
    log=WORK/(out.name+'.log')
    with log.open('w') as f:
        process=subprocess.Popen(command,cwd=ROOT,env=dict(ENV,CUDA_VISIBLE_DEVICES='0'),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='');f.write(line);f.flush()
        code=process.wait()
    assert code==0, f'Job failed. Download {log} and the partial ZIP for diagnosis.'
    return out
SMOKE=execute(True)
print('Smoke passed. The next cell starts a fresh full job.')


In [ ]:
OUT=execute(False)
from IPython.display import display,FileLink
display(FileLink(str(OUT.with_suffix('.zip'))))
print((OUT/'summary.json').read_text())
